In [1]:
import numpy as np
import scipy.sparse as sp
import math
import random

# =====================================================================
# クラス1: APM部品ビルダー (非可換ブレーカー抽出・局所最適化)
# =====================================================================
class APMSCHGP_UnifiedBuilder:
    def __init__(self, r1=3, n1=4, r2=3, n2=4, m1=2, m2=2, L1=12, L2=12, J=3, P=120):
        self.r1, self.n1 = r1, n1
        self.r2, self.n2 = r2, n2
        self.m1, self.m2 = m1, m2
        self.L1, self.L2 = L1, L2
        self.J = J
        self.P = P

        self.base_A = np.ones((self.r1, self.n1), dtype=np.int8)
        self.commutative_pool = []
        self.breaker_apm = None  

        random.seed(42)
        np.random.seed(42)

    def check_commutativity(self, apm1, apm2):
        a1, b1 = apm1
        a2, b2 = apm2
        return (b1 * (a2 - 1)) % self.P == (b2 * (a1 - 1)) % self.P

    def phase3_apm_algebraic_search(self):
        valid_a = [a for a in range(2, self.P) if math.gcd(a, self.P) == 1]
        all_apms = [(a, b) for a in valid_a for b in range(self.P)]
        random.shuffle(all_apms)

        # 1. 互いに可換なプールを探す
        pool_size = self.r1 * self.n1 
        for anchor in all_apms:
            pool = [anchor]
            for candidate in all_apms:
                if candidate == anchor: continue
                if all(self.check_commutativity(candidate, p) for p in pool):
                    pool.append(candidate)
                if len(pool) == pool_size:
                    self.commutative_pool = pool
                    break
            if len(self.commutative_pool) == pool_size: break
        
        # 2. ブレーカーAPMを探す
        for candidate in all_apms:
            if all(not self.check_commutativity(candidate, p) for p in self.commutative_pool):
                self.breaker_apm = candidate
                print(f">> [Builder] 非可換ブレーカーAPMを抽出: a={candidate[0]}, b={candidate[1]}")
                break

    def generate_apm_matrix(self, apm):
        a, b = apm
        mat = np.zeros((self.P, self.P), dtype=np.int8)
        for x in range(self.P): mat[(a * x + b) % self.P, x] = 1
        return mat

    def expand_base_matrix(self, assignments):
        H = np.zeros((self.r1 * self.P, self.n1 * self.P), dtype=np.int8)
        for i in range(self.r1):
            for j in range(self.n1):
                H[i*self.P:(i+1)*self.P, j*self.P:(j+1)*self.P] = self.generate_apm_matrix(assignments[i][j])
        return H

    def count_4_cycles(self, H):
        overlap = H @ H.T
        np.fill_diagonal(overlap, 0)
        return np.sum(overlap * (overlap - 1)) // 4

    def build(self):
        print("=== ステップ1: APM部品の構成と最適化 ===")
        self.phase3_apm_algebraic_search()
        
        current_assignments = [[random.choice(self.commutative_pool) for _ in range(self.n1)] for _ in range(self.r1)]
        best_H = self.expand_base_matrix(current_assignments)
        best_cycles = self.count_4_cycles(best_H)

        for _ in range(100):
            i, j = random.randint(0, self.r1 - 1), random.randint(0, self.n1 - 1)
            old_apm, new_apm = current_assignments[i][j], random.choice(self.commutative_pool)
            if old_apm == new_apm: continue
            current_assignments[i][j] = new_apm
            test_H = self.expand_base_matrix(current_assignments)
            cycles = self.count_4_cycles(test_H)
            if cycles < best_cycles:
                best_cycles, best_H = cycles, test_H
            else:
                current_assignments[i][j] = old_apm
        print(f">> [Builder] 最適化完了: 残存する4-サイクル数 = {best_cycles}")
        return best_H

# =====================================================================
# クラス2: 2D-SC HGP アセンブラ (Breaking the Orthogonality Barrier)
# =====================================================================
class Full_2DSCHGP_Assembler:
    def __init__(self, builder_instance, H_A_dense):
        self.b = builder_instance
        self.blocks_A = self._extract_blocks(H_A_dense, self.b.r1, self.b.n1)
        self.blocks_B = self.blocks_A 
        self.breaker_matrix = sp.csr_matrix(self.b.generate_apm_matrix(self.b.breaker_apm))
        
    def _extract_blocks(self, H_dense, r, n):
        blocks = []
        for i in range(r):
            row_blocks = []
            for j in range(n):
                block = H_dense[i*self.b.P:(i+1)*self.b.P, j*self.b.P:(j+1)*self.b.P]
                row_blocks.append(sp.csr_matrix(block))
            blocks.append(row_blocks)
        return blocks

    def build_sc_blocks(self, blocks, r, n):
        """L1 x L2 のトーラス空間上にブロックを展開し、境界にブレーカーAPMを注入する"""
        sc_blocks = []
        S_dim = self.b.L1 * self.b.L2 * self.b.P
        for i in range(r):
            row_blocks = []
            for j in range(n):
                if blocks[i][j].nnz == 0:
                    row_blocks.append(sp.csr_matrix((S_dim, S_dim), dtype=np.int8))
                    continue
                
                u, v = np.random.randint(0, self.b.m1 + 1), np.random.randint(0, self.b.m2 + 1)
                
                grid_mat = sp.lil_matrix((S_dim, S_dim), dtype=np.int8)
                normal_block = blocks[i][j]
                breaker_block = self.breaker_matrix

                for y in range(self.b.L1):
                    for x in range(self.b.L2):
                        ty = (y + u) % self.b.L1
                        tx = (x + v) % self.b.L2

                        row_start = (y * self.b.L2 + x) * self.b.P
                        col_start = (ty * self.b.L2 + tx) * self.b.P

                        # 上部の J ブロック分（行）をアクティブとし、それより下部にブレーカーを注入
                        # x（列方向）については全てアクティブ領域として扱う
                        if y >= self.b.J:
                            grid_mat[row_start:row_start+self.b.P, col_start:col_start+self.b.P] = breaker_block
                        else:
                            grid_mat[row_start:row_start+self.b.P, col_start:col_start+self.b.P] = normal_block

                row_blocks.append(grid_mat.tocsr())
            sc_blocks.append(row_blocks)
        return sc_blocks

    def build_matrices(self):
        print("\n=== ステップ2: トーラス空間への展開とブレーカーAPMの注入 ===")
        A_sc_blocks = self.build_sc_blocks(self.blocks_A, self.b.r1, self.b.n1)
        B_sc_blocks = self.build_sc_blocks(self.blocks_B, self.b.r2, self.b.n2)
        
        print("=== ステップ3: ハイパーグラフ積 (HGP) による親行列の構成 ===")
        rows_X = self.b.n2 * self.b.r1
        cols_X = self.b.n2 * self.b.n1 + self.b.r2 * self.b.r1
        HX_bmat = [[None for _ in range(cols_X)] for _ in range(rows_X)]
        
        for i in range(self.b.n2):
            for u in range(self.b.r1):
                for v in range(self.b.n1):
                    HX_bmat[i * self.b.r1 + u][i * self.b.n1 + v] = A_sc_blocks[u][v]
        offset_X = self.b.n2 * self.b.n1
        for i in range(self.b.n2):
            for j in range(self.b.r2):
                for u in range(self.b.r1):
                    HX_bmat[i * self.b.r1 + u][offset_X + j * self.b.r1 + u] = B_sc_blocks[j][i].T
        H_X_full = sp.bmat(HX_bmat, format='csr')
        
        rows_Z = self.b.r2 * self.b.n1
        cols_Z = self.b.n2 * self.b.n1 + self.b.r2 * self.b.r1
        HZ_bmat = [[None for _ in range(cols_Z)] for _ in range(rows_Z)]
        
        for i in range(self.b.r2):
            for j in range(self.b.n2):
                for u in range(self.b.n1):
                    HZ_bmat[i * self.b.n1 + u][j * self.b.n1 + u] = B_sc_blocks[i][j]
        offset_Z = self.b.n2 * self.b.n1
        for i in range(self.b.r2):
            for j in range(self.b.r2):
                if i == j:
                    for u in range(self.b.n1):
                        for v in range(self.b.r1):
                            HZ_bmat[i * self.b.n1 + u][offset_Z + i * self.b.r1 + v] = A_sc_blocks[v][u].T
        H_Z_full = sp.bmat(HZ_bmat, format='csr')

        print("=== ステップ4: 非可換検査行の破棄（Breaking the Orthogonality Barrier）===")
        print(f"  [展開時] H_X 親行列: {H_X_full.shape[0]} 行 x {H_X_full.shape[1]} 列")
        print(f"  [展開時] H_Z 親行列: {H_Z_full.shape[0]} 行 x {H_Z_full.shape[1]} 列")
        
        # 1. 直交性を破壊した親行列同士の積を計算
        O = H_X_full @ H_Z_full.T
        O.data = O.data % 2
        O.eliminate_zeros()
        
        # 2. 反可換となってしまった行を特定
        broken_X = np.unique(O.nonzero()[0])
        broken_Z = np.unique(O.nonzero()[1])
        
        print(f"  >> ブレーカーによって直交性が破壊された X-check: {len(broken_X)} 個")
        print(f"  >> ブレーカーによって直交性が破壊された Z-check: {len(broken_Z)} 個")
        
        # 3. 壊れた検査行を捨てる
        valid_X = np.setdiff1d(np.arange(H_X_full.shape[0]), broken_X)
        valid_Z = np.setdiff1d(np.arange(H_Z_full.shape[0]), broken_Z)
        
        H_X_final = H_X_full[valid_X, :]
        H_Z_final = H_Z_full[valid_Z, :]
        
        print(f"  [最終] H_X 行列: {H_X_final.shape[0]} 行 x {H_X_final.shape[1]} 列")
        print(f"  [最終] H_Z 行列: {H_Z_final.shape[0]} 行 x {H_Z_final.shape[1]} 列")
        
        # 最終確認
        O_final = H_X_final @ H_Z_final.T
        O_final.data = O_final.data % 2
        O_final.eliminate_zeros()
        
        if O_final.nnz == 0 and H_X_final.shape[0] > 0:
            print("   [PASS] 素晴らしい！ 最終的な H_X と H_Z は完全に直交しています。")
        else:
            print("   [FAIL] 直交性の破綻、または行数が0です。")

        return H_X_final, H_Z_final, (O_final.nnz == 0)

# ---------------------------------------------------------
# メイン実行ブロック
# ---------------------------------------------------------
if __name__ == "__main__":
    config = {
        'r1': 3, 'n1': 4, 'r2': 3, 'n2': 4,
        'm1': 2, 'm2': 2,
        'L1': 12, 'L2': 12,
        'J': 3,
        'P': 768 # テスト用サイズ。必要に応じて768などに変更
    }
    
    builder = APMSCHGP_UnifiedBuilder(**config)
    H_active_optimized = builder.build()
    
    assembler = Full_2DSCHGP_Assembler(builder, H_active_optimized)
    H_X_final, H_Z_final, is_ortho = assembler.build_matrices()

=== ステップ1: APM部品の構成と最適化 ===
>> [Builder] 非可換ブレーカーAPMを抽出: a=545, b=506
>> [Builder] 最適化完了: 残存する4-サイクル数 = 80

=== ステップ2: トーラス空間への展開とブレーカーAPMの注入 ===
=== ステップ3: ハイパーグラフ積 (HGP) による親行列の構成 ===
=== ステップ4: 非可換検査行の破棄（Breaking the Orthogonality Barrier）===
  [展開時] H_X 親行列: 1327104 行 x 2764800 列
  [展開時] H_Z 親行列: 1327104 行 x 2764800 列
  >> ブレーカーによって直交性が破壊された X-check: 663432 個
  >> ブレーカーによって直交性が破壊された Z-check: 608256 個
  [最終] H_X 行列: 663672 行 x 2764800 列
  [最終] H_Z 行列: 718848 行 x 2764800 列
   [PASS] 素晴らしい！ 最終的な H_X と H_Z は完全に直交しています。


In [2]:
import numpy as np
import scipy.sparse as sp

def save_matrix_as_alist(matrix, filename):
    """
    scipy.sparse行列をAListフォーマットでテキストファイルとして保存する。
    """
    # 処理の効率化のため、行アクセス用(CSR)と列アクセス用(CSC)の両方を準備する
    H_csr = sp.csr_matrix(matrix)
    H_csc = H_csr.tocsc()
    M, N = H_csr.shape # M: 検査数(行), N: 量子ビット数(列)
    
    # 各行・各列の重み（非ゼロ要素の数）を計算
    row_weights = np.diff(H_csr.indptr)
    col_weights = np.diff(H_csc.indptr)
    
    max_row_weight = np.max(row_weights) if len(row_weights) > 0 else 0
    max_col_weight = np.max(col_weights) if len(col_weights) > 0 else 0
    
    print(f"[{filename}] の書き出しを開始... (サイズ: {M}x{N})")
    
    with open(filename, 'w') as f:
        # 1行目: N(列数) M(行数)
        f.write(f"{N} {M}\n")
        # 2行目: 最大列重み 最大行重み
        f.write(f"{max_col_weight} {max_row_weight}\n")
        # 3行目: 各列の重み (スペース区切り)
        f.write(" ".join(map(str, col_weights)) + "\n")
        # 4行目: 各行の重み (スペース区切り)
        f.write(" ".join(map(str, row_weights)) + "\n")
        
        # 第1ブロック: 各列(量子ビット)がどの行(チェック)に参加しているか (1-based index)
        for j in range(N):
            start, end = H_csc.indptr[j], H_csc.indptr[j+1]
            indices = H_csc.indices[start:end] + 1 # C++等の標準に合わせて1始まりにする
            f.write(" ".join(map(str, indices)) + "\n")
            
        # 第2ブロック: 各行(チェック)がどの列(量子ビット)を含んでいるか (1-based index)
        for i in range(M):
            start, end = H_csr.indptr[i], H_csr.indptr[i+1]
            indices = H_csr.indices[start:end] + 1
            f.write(" ".join(map(str, indices)) + "\n")
            
    print(f" >> 保存完了: {filename}\n")

# ---------------------------------------------------------
# 実行ブロック
# H_X_final, H_Z_final がメモリ上に存在することを前提とする
# ---------------------------------------------------------
if 'H_X_final' in globals() and 'H_Z_final' in globals():
    # C++シミュレータに読み込ませるためのAListファイルを生成
    save_matrix_as_alist(H_X_final, "H_X.alist")
    save_matrix_as_alist(H_Z_final, "H_Z.alist")
else:
    print("エラー: 行列が定義されていない。")

[H_X.alist] の書き出しを開始... (サイズ: 663672x2764800)
 >> 保存完了: H_X.alist

[H_Z.alist] の書き出しを開始... (サイズ: 718848x2764800)
 >> 保存完了: H_Z.alist



In [4]:
import numpy as np
import scipy.sparse as sp
import time
import os
from ldpc import BpOsdDecoder

def load_matrix_from_alist(filename):
    """
    AListフォーマットのファイルを読み込み、scipy.sparse.csr_matrixを生成する。
    """
    print(f">> {filename} を読み込み中...")
    with open(filename, 'r') as f:
        lines = f.read().splitlines()

    # 修正点: 空行（重み0の行/列）を削除せず、空白文字の除去のみを行う
    lines = [line.strip() for line in lines]

    # 1行目: N(列数=量子ビット数), M(行数=スタビライザー数)
    N, M = map(int, lines[0].split())
    
    # 行の接続データは、ヘッダ(4行) + 列データ(N行) の後から始まる
    row_start_idx = 4 + N
    
    indptr = [0]
    indices = []
    data = []

    for i in range(M):
        # 空行（重み0の行）の場合は split() が空リストを返すため安全に処理される
        row_data = list(map(int, lines[row_start_idx + i].split()))
        
        # AListは 1-based index であり、0はパディングを意味するため除外して 0-based に戻す
        valid_indices = [idx - 1 for idx in row_data if idx > 0]
        
        indices.extend(valid_indices)
        data.extend([1] * len(valid_indices))
        indptr.append(len(indices))

    # CSR行列として構築
    H = sp.csr_matrix((data, indices, indptr), shape=(M, N), dtype=np.int8)
    print(f"   完了: {M} 行 x {N} 列")
    return H

class CSSCodeEvaluator:
    """
    H_X と H_Z の両方を用いた完全なCSS量子符号の BP-OSD シミュレータ
    """
    def __init__(self, H_X, H_Z, error_rate=0.04):
        self.H_X = H_X
        self.H_Z = H_Z
        self.error_rate = error_rate
        
        self.N_qubits = self.H_X.shape[1]
        
        print(f"\n--- CSS BP-OSD デコーダの初期化 ---")
        print(f"量子ビット数 (N) : {self.N_qubits}")
        print(f"Xスタビライザー数: {self.H_X.shape[0]}")
        print(f"Zスタビライザー数: {self.H_Z.shape[0]}")
        
        # 1. Zエラーを検出・修正するデコーダ (H_X を使用)
        self.decoder_Z = BpOsdDecoder(
            self.H_X,
            error_rate=self.error_rate,
            max_iter=50,
            bp_method="ms",
            osd_method="osd_cs",
            osd_order=10
        )
        
        # 2. Xエラーを検出・修正するデコーダ (H_Z を使用)
        self.decoder_X = BpOsdDecoder(
            self.H_Z,
            error_rate=self.error_rate,
            max_iter=50,
            bp_method="ms",
            osd_method="osd_cs",
            osd_order=10
        )

    def run_simulation(self, trials=1000):
        print(f"\n--- CSS モンテカルロ・シミュレーション開始 ---")
        print(f"物理エラー率 (p) : {self.error_rate}")
        print(f"試行回数         : {trials} 回\n")
        
        success_count = 0
        fail_count = 0
        start_time = time.time()

        for t in range(trials):
            # XエラーとZエラーを独立して発生
            err_X = np.random.binomial(1, self.error_rate, self.N_qubits)
            err_Z = np.random.binomial(1, self.error_rate, self.N_qubits)
            
            if np.sum(err_X) == 0 and np.sum(err_Z) == 0:
                success_count += 1
                continue
            
            # Zエラーの復号
            syndrome_X = self.H_X.dot(err_Z) % 2
            guess_Z = self.decoder_Z.decode(syndrome_X)
            
            # Xエラーの復号
            syndrome_Z = self.H_Z.dot(err_X) % 2
            guess_X = self.decoder_X.decode(syndrome_Z)
            
            # 収束判定（デコーダが矛盾なくシンドロームを消去できたか）
            res_Z = (err_Z + guess_Z) % 2
            res_X = (err_X + guess_X) % 2
            
            check_Z = self.H_X.dot(res_Z) % 2
            check_X = self.H_Z.dot(res_X) % 2
            
            if np.all(check_Z == 0) and np.all(check_X == 0):
                success_count += 1
            else:
                fail_count += 1
                
            if (t + 1) % 100 == 0:
                print(f"  [{t + 1:4d} / {trials}] 完了... (現在の失敗数: {fail_count})")

        elapsed_time = time.time() - start_time
        logical_error_rate = fail_count / trials

        print(f"\n=== シミュレーション結果 ===")
        print(f"総試行回数   : {trials}")
        print(f"復号成功     : {success_count} (※シンドローム収束)")
        print(f"復号失敗     : {fail_count}")
        print(f"論理エラー率 : {logical_error_rate:.6f}")
        print(f"実行時間     : {elapsed_time:.2f} 秒")

# ---------------------------------------------------------
# 実行ブロック
# ---------------------------------------------------------
if __name__ == "__main__":
    file_X = "H_X.alist"
    file_Z = "H_Z.alist"
    
    if os.path.exists(file_X) and os.path.exists(file_Z):
        # ファイルから行列を復元
        H_X_loaded = load_matrix_from_alist(file_X)
        H_Z_loaded = load_matrix_from_alist(file_Z)
        
        # シミュレータにロードして実行 (p=4%)
        css_evaluator = CSSCodeEvaluator(H_X_loaded, H_Z_loaded, error_rate=0.04)
        css_evaluator.run_simulation(trials=1000)
    else:
        print(f"エラー: {file_X} または {file_Z} が見つからない。先にAListファイルを生成する必要がある。")

>> H_X.alist を読み込み中...
   完了: 663672 行 x 2764800 列
>> H_Z.alist を読み込み中...


: 